# День 12: финальные модели на всех размеченных данных (~1.8 ч)
Day11: смесь 0.5·ft2_e0 + 0.2·gru_full + 0.3·reg128m — **0.65612** на лидерборде. Прирост на валидации (+0.0011) перешёл на лидерборд почти один в один (+0.0010), значит дополнительные данные действительно помогают, а не просто «подсказывают» валидацию.

Последний шаг: обучить те же модели **на всех размеченных данных** — train + вся валидация, включая 600 последовательностей `valid_sub` (ещё +5% к тому, что было). Рецепты и веса смеси уже проверены, поэтому новые модели не выбираются по валидации, а берутся по фиксированному правилу:
1. **ft_all** — ft_swa + 1 эпоха на всех данных, lr 1e-4 (рецепт ft2_e0; ~12 мин);
2. **reg128_all** — GRU-128 с нуля, рецепт reg128m (dropout 0.3, шум 0.1, wd 2e-2, сид 2, 8 эпох), берётся последняя эпоха (~1.4 ч);
3. **gru_full** — без изменений; веса смеси те же: 0.5 / 0.2 / 0.3.

**Важно:** эти модели видели `valid_sub`, поэтому строки `VALID` в логах и `WP on the validation mask` при сборке **завышены** — это не оценка, а только проверка, что обучение не сломалось. Сравнение — только по лидерборду: если новый сабмит окажется ниже 0.65612, финальным оставьте 0.65612.

Если времени мало: после шага 4 можно сразу собрать вариант с reg128m вместо reg128_all (ячейка 6, `QUICK = True`).

## 1. GPU + Drive + код + numba как у скорера

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!pip -q install "numba==0.67.0" onnx 2>&1 | grep -v "dependency resolver\|requires numba\|incompatible" || true
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2
!grep -q "no-cv" ensemble_eval.py && echo "code: OK" || echo "OLD CODE on Drive - upload the new day2 folder"

## 2. Данные: train.parquet и valid.parquet (~6 мин) + `valid_sub` для логов и сборки (~1.5 мин)

In [ ]:
import glob
URL = 'https://files.wundernn.io/wnn_connectome_starterpack.tar.gz'
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y aria2 > /dev/null 2>&1
!aria2c -c -x 16 -s 16 -k 1M --file-allocation=none --summary-interval=60 --console-log-level=warn -d /content -o pack.tar.gz $URL
!mkdir -p /content/sp && tar -xzf /content/pack.tar.gz -C /content/sp --wildcards '*datasets/train.parquet' '*datasets/valid.parquet' 2>&1 | grep -v LIBARCHIVE
!rm -f /content/pack.tar.gz
TRAIN_PQ = glob.glob('/content/sp/**/datasets/train.parquet', recursive=True)[0]
VALID_PQ = glob.glob('/content/sp/**/datasets/valid.parquet', recursive=True)[0]
HELD = W + '/colab_data/valid_sub.parquet'
!python prepare_data.py --src $HELD --out /content/data/valid

## 3. ft_all: ft_swa + 1 эпоха на train и всей валидации (~12 мин)
В начале лога должно быть `train seqs 12480 ... + 1873 of 1873 sequences`, а `VALID (start ...)` — 0.6668 (это ft_swa).

In [ ]:
FT_SWA = '+'.join(W + f'/runs/ft_baseline_full/epoch_{k}.pt' for k in range(2, 6))
!python train.py --train-parquet $TRAIN_PQ --extra-parquet $VALID_PQ --valid-dir /content/data/valid --out $W/runs/ft_all \
    --arch baseline --init-from $FT_SWA --keep-epochs --epochs 1 --batch 32 --chunk 1000 --lr 1e-4 --mse-weight 0.1 \
    --loader-workers 4

## 4. reg128_all: GRU-128 с нуля на train и всей валидации (~1.4 ч)
Если сессия оборвалась: ячейки 1–2, затем эта же ячейка с `--resume` в конце команды.

In [ ]:
!python train.py --train-parquet $TRAIN_PQ --extra-parquet $VALID_PQ --valid-dir /content/data/valid --out $W/runs/reg128_all \
    --arch gru --hidden 128 --epochs 8 --batch 64 --chunk 1000 --lr 2e-3 --keep-epochs --seed 2 \
    --dropout 0.3 --input-noise 0.1 --wd 2e-2 --loader-workers 4

## 5. Сборка смеси 0.5·ft_all + 0.2·gru_full + 0.3·reg128_all, проверка, скорость, скачивание (~15 мин)
`QUICK = True` — если шаг 4 не успели: вместо reg128_all берётся reg128m из day10.

In [ ]:
import os
QUICK = False
REG = W + ('/runs/reg128m/best.pt' if QUICK else '/runs/reg128_all/epoch_7.pt')
FT = W + '/runs/ft_all/epoch_0.pt'
for p in (FT, REG, W + '/runs/gru_full/best.pt'):
    assert os.path.exists(p), f'not found: {p}'
OUT = W + '/submissions/ens12_all' + ('_quick' if QUICK else '')
REF = W + '/submissions/gru128x2_v1'
SPEC = f'--model "ft_all={FT}@0.5" --model "gru_full={W}/runs/gru_full/best.pt@0.2" --model "reg={REG}@0.3"'
!python build_ensemble.py --valid-dir /content/data/valid --out $OUT $SPEC
!python check_submission.py {OUT}.zip --data $HELD --seqs 3 --rows 20000 --ref-dir $REF
!python bench_speed.py $REF $OUT --rows 5000 --repeats 5
from google.colab import files
files.download(OUT + '.zip')

## Что дальше
Отправьте zip и сравните с 0.65612. Выше — этот сабмит и делаем финальным; ниже или так же — финальным оставьте 0.65612 (ens11A).